# 09 · Data size, training design and label window

Three research questions about the *data* rather than the model:

| Run | Question |
|---|---|
| DS-01 | **Learning curves:** how does each model family improve with more rows? Does boosting overtake logistic regression? (Benchmarking doc: flexible models need ~10× more data.) |
| DS-02 | **Training design:** does a model trained on call-anchored rows rank patients as well as one trained on fixed-date rows? |
| DS-04 | **Label window:** 30 vs 60 vs 90 days after the scoring date. |

(The training-*source* question — synthetic vs real — needs the real rows, so it is answered once in notebook 11.)

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import loguniform, uniform, randint
from salud_pipeline.phase2.propensity import modeling as M
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200); pd.set_option("display.precision", 3)

DESIGN = "fixed"   # training rows: "fixed" (one per patient on the scoring date; notebook 00 §6) or "call"
WORLD = "salud"    # synthetic world: "salud" (tuned to Salud's data, salud-synth-calibrate) or "original"
dev, holdout, real = M.load_data(design=DESIGN, world=WORLD)
SPLITS = M.cv_splits(dev)          # the same 25 folds for every run in every notebook -> paired comparisons
NB = "09"
def run(rid, pipe, fs, desc, **kw):
    return M.run_experiment(rid, pipe, dev, fs, notebook=NB, description=desc, splits=SPLITS, **kw)
print(f"{DESIGN} design: {len(dev):,} training rows, pay rate {dev.paid_30d.mean():.2f}; 25 folds")

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
F = M.FS_TRANSFER
MODELS = {
    "Logistic regression (L2)": lambda: M.pipeline(M.preprocess("z"), M.logreg(C=0.1)),
    "Scorecard (monotone WoE)": lambda: M.pipeline([("woe", M.WoE(monotonic=True))], M.logreg(C=1.0)),
    "LightGBM (monotone)": lambda: M.pipeline(M.preprocess("none"), M.lgbm(F, monotone=True, num_leaves=7, learning_rate=.03,
                                     n_estimators=300, min_child_samples=20, subsample=.8, subsample_freq=1, colsample_bytree=.8)),
    "Extra trees": lambda: M.pipeline(M.preprocess("none"), ExtraTreesClassifier(400, min_samples_leaf=10, max_features=.5, random_state=M.SEED, n_jobs=-1)),
    "Neural network": lambda: M.pipeline(M.preprocess("log_z"), MLPClassifier((16,), alpha=1.0, early_stopping=True, max_iter=2000, random_state=M.SEED)),
}

## DS-01 · Learning curves (10 grouped folds; the training part is subsampled to n rows, the test part is untouched)

In [ ]:
sizes = [50, 100, 200, 400, 650, None]
X, y = M.xy(dev, F); rng = np.random.default_rng(M.SEED); rows = []
for s in SPLITS[:10]:
    for n in sizes:
        tr = s.train if n is None or n >= len(s.train) else rng.choice(s.train, n, replace=False)
        if len(set(y[tr])) < 2: continue
        for name, mk in MODELS.items():
            p = clone(mk()).fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[s.test])[:, 1]
            rows.append({"model": name, "n": len(tr), "auc": roc_auc_score(y[s.test], p)})
lc = pd.DataFrame(rows)
lc.to_csv(M.RESULTS.parent / "learning_curves.csv", index=False)
curve = lc.groupby(["model", "n"]).auc.agg(["mean", "sem"]).reset_index()
plt.figure(figsize=(9, 4.5))
for name, g in curve.groupby("model"):
    plt.errorbar(g.n, g["mean"], yerr=1.96 * g["sem"], marker="o", capsize=3, label=name)
plt.xscale("log"); plt.xlabel("training rows (log scale)"); plt.ylabel("AUC on the untouched test fold"); plt.legend(); plt.title("Learning curves"); plt.show()
curve.pivot(index="n", columns="model", values="mean").round(3)

## DS-02 · Training design: call-anchored rows vs fixed-date rows, both scored on fixed-date test patients

In [ ]:
call_dev, _, _ = M.load_data(design="call", world=WORLD)
Xf, yf = M.xy(dev, F); rows = []
for s in SPLITS:
    test_pat = set(dev.guarantor_id.iloc[s.test]); train_pat = set(dev.guarantor_id.iloc[s.train])
    tr_call = call_dev[call_dev.guarantor_id.isin(train_pat)]          # same training patients, their call rows
    Xc, yc = M.xy(tr_call, F)
    for name in ["Logistic regression (L2)", "Scorecard (monotone WoE)"]:
        p_fixed = clone(MODELS[name]()).fit(Xf.iloc[s.train], yf[s.train]).predict_proba(Xf.iloc[s.test])[:, 1]
        p_call = clone(MODELS[name]()).fit(Xc, yc).predict_proba(Xf.iloc[s.test])[:, 1]
        rows += [{"model": name, "trained on": "fixed-date rows", "auc": roc_auc_score(yf[s.test], p_fixed)},
                 {"model": name, "trained on": "call rows (same patients)", "auc": roc_auc_score(yf[s.test], p_call)}]
pd.DataFrame(rows).groupby(["model", "trained on"]).auc.agg(["mean", "std"]).round(3)

## DS-04 · Label window (fixed-date design rebuilt with 30, 60 and 90-day outcomes)

In [ ]:
import json
from salud_pipeline.paths import PHASE2_WORLDS
from salud_pipeline.phase2.propensity.sources import load_synthetic
from salud_pipeline.phase2.propensity.snapshots import build, fixed_date_world, open_balance
W = load_synthetic(PHASE2_WORLDS[WORLD])
cut = pd.Timestamp(json.loads((PHASE2_WORLDS[WORLD] / "manifest.json").read_text())["config"]["prediction_cutoff"])
rows = []
for days in (30, 60, 90):
    snap = open_balance(build(fixed_date_world(W, cut), window_days=days))   # the world's outcome window is complete by design
    for b in snap.select_dtypes("bool"): snap[b] = snap[b].astype(int)
    for name in ["Logistic regression (L2)", "Scorecard (monotone WoE)"]:
        f = M.run_experiment(f"DS-04-{days}", MODELS[name](), snap, F, notebook=NB, description=f"{name}, {days}-day label",
                             splits=M.cv_splits(snap), log=False, verbose=False)
        rows.append({"window (days)": days, "model": name, "pay rate": snap.paid_30d.mean(), "AUC": f.auc.mean(), "SD": f.auc.std()})
pd.DataFrame(rows).round(3)

## Findings (Salud-tuned world)
- **Learning curves:**
  - Extra trees, LightGBM, the scorecard and logistic regression already reach 0.83–0.86 with 50 patients and plateau by about 400.
  - The neural network fails below 400 rows (AUC < 0.5 at n ≤ 200, a collapse caused by early stopping on tiny validation sets).
  - More synthetic rows are not the bottleneck.
- **Row design:** training on call rows or on fixed-date rows for the same patients gives the same AUC (LR 0.861 both; scorecard 0.874 against 0.871).
- **Label window:** 30 days is the most predictable (scorecard 0.861, against 0.831 at 60 days and 0.801 at 90). Keep 30 days.